## Extract Raven-annotated units as WAV clips
---
Parse Raven selection tables paired with WAV files and export labeled (and unlabeled) units as individual clips. These clips are the input to `train_raven_units.ipynb`.

Author: Danelle Cline dcline@mbari.org

### Set paths
Choose the config YAML, dataset directory of WAV files plus Raven tables, and output directory for exported unit clips.

In [1]:
from pathlib import Path
config_yaml_path =  "../config.yaml"
output_path = Path("output_hb")
dataset_path = Path("dataset_hb")
output_path.mkdir(parents=True, exist_ok=True)

### Load configuration
Load and verify the config. Window length for each exported clip comes from `perch_audio_seconds`.

In [2]:
from perchtopic.config import Config
try:
    config = Config(config_yaml_path, output_path)
    config.verify()
except Exception as e:
    print(e)



CONFIG
wav_path: ../config.yaml 
output_path: output_hb/doc
model_path: output_hb/model
perch_hop_seconds: 0.5
perch_audio_seconds: 5.0
perch_window_fill: fill
PERCH_TIME_BIN_SECONDS: 5


### Export unit clips
For each WAV with a matching `.selections.txt` table, parse Raven annotations and write unit WAVs under `output_path/units`, including unlabeled regions.

In [3]:
from perchtopic.raven_parser import RavenParser
import soundfile as sf
wav_out_root = output_path / "units"
wav_out_root.mkdir(parents=True, exist_ok=True)

for f in dataset_path.rglob("*.wav"):
    anno_file = Path(f).with_suffix(".selections.txt")
    if not anno_file.exists():
        print(f"Skipping {f}")
        continue
    info = sf.info(f.as_posix())
    print(info)
    window_seconds = config.perch_audio_seconds
    parser = RavenParser(anno_file, info.frames, info.samplerate, window_seconds, config)
    RavenParser.export_wavs(parser, f, wav_out_root / f.stem, config, include_unlabeled=True)

print("Done")


dataset_hb/MARS_20161221_000046_SongSession_32kHz_HPF5Hz.wav
samplerate: 32000 Hz
channels: 1
duration: 4:4e+01:12.330 h
format: WAV (Microsoft) [WAV]
subtype: Signed 16 bit PCM [PCM_16]
Reading dataset_hb/MARS_20161221_000046_SongSession_32kHz_HPF5Hz.selections.txt
Found 5470 detections in dataset_hb/MARS_20161221_000046_SongSession_32kHz_HPF5Hz.selections.txt
Sampled 25 background windows
Wrote 5545 clips to output_hb/units/MARS_20161221_000046_SongSession_32kHz_HPF5Hz/<label>/
Done
